# 02 — BART Text Summarization

This notebook demonstrates abstractive summarization with:

`facebook/bart-large-cnn`

The checkpoint is fine-tuned on CNN/DailyMail.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
MODEL_NAME = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Device:", device)

In [4]:
article = '''
Artificial intelligence is changing the way organizations work with information.
Modern Transformer models can read large amounts of text and generate concise
summaries that preserve important ideas. Text summarization is useful in news,
research, business intelligence, customer support and document management.
However, long documents can exceed a model's input limit, so practical systems
often divide documents into smaller sections before generating summaries.
'''

In [5]:
inputs = tokenizer(
    article,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)
inputs = {key: value.to(device) for key, value in inputs.items()}

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_length=100,
        min_length=25,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True,
        no_repeat_ngram_size=3
    )

summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(summary)

Modern Transformer models can read large amounts of text and generate concise summaries that preserve important ideas. Text summarization is useful in news, research, business intelligence, customer support and document management.


## Generation Settings

- `num_beams=4`: beam search explores multiple candidate sequences.
- `max_length`: controls the upper output length.
- `min_length`: avoids an extremely short summary.
- `no_repeat_ngram_size=3`: reduces repeated phrases.
- `length_penalty`: encourages concise generation.

In [6]:
def summarize(article):
    inputs = tokenizer(
        article,
        return_tensors="pt",
        truncation=True,
        max_length=850
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        ids = model.generate(
            **inputs,
            max_length=150,
            min_length=35,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    return tokenizer.decode(ids[0], skip_special_tokens=True)

print(summarize(article))

Modern Transformer models can read large amounts of text and generate concise summaries that preserve important ideas. Text summarization is useful in news, research, business intelligence, customer support and document management.


## Interview Point

This is **abstractive summarization**. The model can generate wording that is not copied sentence-for-sentence from the input.

For long documents, the Flask application performs sentence-aware chunking and summarizes each chunk before combining the results.